# Solutions — ArrayList / HashMap / HashSet / for-each → Python

Exercise notebook: `04_java_to_python/01_collections_and_iteration.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Removing while iterating

**Working solution**

In [ ]:
def without_cancelled(orders):
    return [o for o in orders if o["status"].upper() != "CANCELLED"]

**Explanation**

`for o in orders: if ...: orders.remove(o)` **skips elements** in Python (the index shifts under the loop). Build a new list instead;
if you truly must mutate in place, use slice assignment: `orders[:] = [o for o in orders if ...]`.

**The bug you'd get by translating literally**

In [ ]:
def buggy(orders):
    for o in orders:
        if o["status"] == "CANCELLED":
            orders.remove(o)        # skips the element right after each removed one
    return orders

print(buggy([{"status": "CANCELLED"}, {"status": "CANCELLED"}, {"status": "OK"}]))  # one CANCELLED survives!

**Complexity:** O(n).

**Common mistakes**

- Mutating the list you iterate over.

## Exercise 2 — 🟢 Easy · HashMap.merge counting → Counter

**Working solution**

In [ ]:
from collections import Counter

def status_counts(orders):
    counts = Counter(o["status"] for o in orders)
    return dict(sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])))

**Explanation**

`Counter` is a dict subclass built for this. It also supports `most_common(n)`, `+`, `-`, and `update()`.

**Complexity:** O(n + k log k).

**Common mistakes**

- Re-implementing counting with `if key in d: d[key] += 1 else: d[key] = 1`.

## Exercise 3 — 🟡 Medium · computeIfAbsent grouping → defaultdict

**Working solution**

In [ ]:
from collections import defaultdict

def group_by(items, key):
    groups = defaultdict(list)
    for item in items:
        groups[key(item)].append(item)
    return dict(groups)

**Explanation**

`defaultdict(list)` is `computeIfAbsent(k, k -> new ArrayList<>())` built into the type. Convert to `dict` before returning so callers
don't accidentally create keys by reading. (`itertools.groupby` only groups **consecutive** items — it needs sorted input.)

**Complexity:** O(n).

**Common mistakes**

- Using `itertools.groupby` on unsorted data and getting split groups.

## Exercise 4 — 🟠 Hard · LinkedHashMap LRU cache → OrderedDict

**Working solution**

In [ ]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity: int):
        if capacity < 1:
            raise ValueError("capacity must be >= 1")
        self.capacity = capacity
        self._data = OrderedDict()
        self.hits = 0
        self.misses = 0

    def get(self, key, default=None):
        if key not in self._data:
            self.misses += 1
            return default
        self.hits += 1
        self._data.move_to_end(key)
        return self._data[key]

    def put(self, key, value):
        self._data[key] = value
        self._data.move_to_end(key)
        if len(self._data) > self.capacity:
            self._data.popitem(last=False)

    def __len__(self):
        return len(self._data)

    def __contains__(self, key):
        return key in self._data

**Explanation**

`OrderedDict.move_to_end` and `popitem(last=False)` are O(1) — the whole LRU in a few lines. Implementing `__len__`/`__contains__`
makes `len(cache)` and `key in cache` work, which is how Python 'implements interfaces'. For function memoization you'd just use
`@functools.lru_cache`.

**Complexity:** O(1) per operation.

**Common mistakes**

- Using a list to track order (O(n) removals).
- Making `in` refresh recency — surprising side effects on a read-only check.

## Challenge — 🔴 Very Hard · TreeMap.floorEntry → bisect: effective-dated FX rates

**Working solution**

In [ ]:
from bisect import bisect_right

def convert_to_inr(txns, rates):
    tables = {}
    for ccy, entries in rates.items():
        ordered = sorted(entries)
        tables[ccy] = ([d for d, _ in ordered], [r for _, r in ordered])
    out = []
    for t in txns:
        if t["currency"] == "INR":
            rate = 1.0
        else:
            if t["currency"] not in tables:
                raise ValueError(f"{t['txn_id']}: unknown currency {t['currency']}")
            dates, values = tables[t["currency"]]
            pos = bisect_right(dates, t["date"]) - 1
            if pos < 0:
                raise ValueError(f"{t['txn_id']}: no rate for {t['currency']} on {t['date']}")
            rate = values[pos]
        out.append({**t, "inr_amount": round(t["amount"] * rate, 2)})
    return out

**Explanation**

`bisect_right(sorted_keys, x) - 1` is `floorKey`. Keep parallel sorted lists (keys, values) — no tree structure needed.
Pre-sorting once and bisecting per txn is O(n log r) total.

**Complexity:** O(r log r + n log r).

**Common mistakes**

- Linear scan of rates per txn (O(n·r)).
- `bisect_left` → wrong when the txn date equals an effective date.